# Notebook 2 — Dataset Preparation

Download StepGame, split into train/eval, format for Unsloth fine-tuning.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import random
from pathlib import Path

from src.dataset import load_stepgame, format_for_training

## Download StepGame

StepGame is available on HuggingFace datasets or from the original repo.
Run the cell below once to download raw splits.

In [ ]:
from datasets import load_dataset

# Load all k-hop splits
ds = load_dataset('sagi21805/StepGame')
print(ds)

In [ ]:
# Inspect a sample
print(ds['train'][0])

## Adapt field names if needed

StepGame fields vary by source. Adjust `story_field`, `question_field`, `answer_field` below.

In [ ]:
STORY_FIELD    = 'story'     # adjust if dataset uses different key
QUESTION_FIELD = 'question'
ANSWER_FIELD   = 'answer'
K_FIELD        = 'k'         # hop count; set None if absent

TRAIN_SIZE = 4000
EVAL_SIZE  = 500
SEED       = 42

raw_data = [
    {
        'story':    ex[STORY_FIELD],
        'question': ex[QUESTION_FIELD],
        'answer':   ex[ANSWER_FIELD],
        'k':        ex.get(K_FIELD),
    }
    for ex in ds['train']
]

random.seed(SEED)
random.shuffle(raw_data)

train_data = raw_data[:TRAIN_SIZE]
eval_data  = raw_data[TRAIN_SIZE:TRAIN_SIZE + EVAL_SIZE]

print(f'Train: {len(train_data)}, Eval: {len(eval_data)}')

In [ ]:
# Format for fine-tuning
train_formatted = [format_for_training(ex) for ex in train_data]

# Save
Path('../data/raw').mkdir(parents=True, exist_ok=True)
Path('../data/processed').mkdir(parents=True, exist_ok=True)
Path('../data/eval').mkdir(parents=True, exist_ok=True)

with open('../data/raw/train.json', 'w') as f:
    json.dump(train_data, f, indent=2)

with open('../data/processed/train_formatted.json', 'w') as f:
    json.dump(train_formatted, f, indent=2)

with open('../data/eval/stepgame_eval.json', 'w') as f:
    json.dump(eval_data, f, indent=2)

print('Saved.')
print('Sample formatted:')
print(train_formatted[0]['full_text'][:500])